<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Post Boot Tasks: Automate Node Configuration at Slice Creation

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook shows you how to add tasks that run automatically when a slice becomes active. Instead of manually uploading files and executing commands after your slice is ready, post boot tasks let you define all configuration steps **before** submitting the slice. FABRIC handles the rest.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Use `node.add_post_boot_upload_directory()` to queue a directory upload that runs automatically after boot
2. Use `node.add_post_boot_execute()` to queue a command that runs automatically after boot
3. Understand the order of execution for post boot tasks
4. Combine uploads and execution into fully automated node configuration
5. Verify that post boot tasks completed successfully

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with creating slices (see [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb))
3. Understand the upload/execute pattern (see [Upload and Execute](../upload_and_execute/upload_and_execute.ipynb))

**Tip:** The `node_tools` directory in this notebook's folder contains the `config_script.sh` script that will be uploaded to each node. Feel free to edit it before running this notebook.

</div>

## Background: What Are Post Boot Tasks?

In previous notebooks, you learned to configure nodes **after** the slice is active by manually calling `upload_file()`, `upload_directory()`, and `execute()`. Post boot tasks automate this entirely.

The key idea is that you define tasks **before** submitting the slice. When the slice becomes active, FABRIC automatically runs these tasks in the order you defined them. Moreover, tasks on different nodes run **in parallel**, so a 4-node slice configures in roughly the same time as a 1-node slice.


There are three post boot task methods:
- **`add_post_boot_upload_file(local, remote)`** -- upload a single file
- **`add_post_boot_upload_directory(local, remote)`** -- upload an entire directory
- **`add_post_boot_execute(command)`** -- run a shell command

Each call appends to a task list that is executed **in order** for each node, but **in parallel** across nodes.

## What We're Building

In this notebook we will create a single compute node with default resources.

<img src="./figs/slice_topology.png" width="40%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Create the Slice with Post Boot Tasks

This is where the magic happens. For each node, we:
1. Add the node to the slice
2. Queue a directory upload (`node_tools/`) that contains our configuration script
3. Queue a command to make the script executable and run it

When `submit()` is called, FABRIC provisions the nodes and then automatically runs all queued tasks.

<div class="fab-info">

**How it works:** The `node_tools` directory is located in the same folder as this notebook. It contains a `config_script.sh` that, by default, installs `tcpdump` using `dnf` and creates an output file. The entire directory is uploaded to each node's home directory, and then the script is executed.

</div>

In [ ]:
# Choose a unique name for the slice
slice_name="MySlice"

# Pick a random site for all nodes (keeps them co-located)
site=fablib.get_random_site()

# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# Add 4 nodes, each with post boot tasks
for i in range(4):
    # Add a node at the chosen site
    node = slice.add_node(name=f"Node{i}", site=site)
    
    # Post boot task 1: Upload the 'node_tools' directory to the node's home directory
    # This copies the entire directory (including config_script.sh) to the remote node
    node.add_post_boot_upload_directory('node_tools','.')
    
    # Post boot task 2: Make the script executable and run it
    # Tasks execute in the order they are added
    node.add_post_boot_execute('chmod +x node_tools/config_script.sh && ./node_tools/config_script.sh')

# Submit the slice -- FABRIC will provision nodes AND run post boot tasks automatically
slice.submit()

<div class="fab-success">

**What just happened?** FABRIC provisioned 4 nodes, uploaded the `node_tools` directory to each one, and executed the configuration script -- all automatically and in parallel across nodes. The `submit()` call blocks until everything is complete.

</div>

## Step 3: Verify Post Boot Tasks Completed

Let's verify that the post boot tasks ran successfully by checking the output file created by the script on each node.

In [ ]:
# Check the output of the config script on each node
# The script creates a file called 'post_boot_output.txt'
for node in slice.get_nodes():
    stdout, stderr = node.execute('echo -n `hostname -s`": " && cat post_boot_output.txt')

## Step 4: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `post_boot_output.txt` not found | Post boot script failed or did not create the file | SSH into the node and check for errors; look at script logic |
| Post boot tasks seem to not run | Slice may have failed during provisioning | Check `slice.list_nodes()` for nodes in error state |
| `node_tools` directory not found on node | Upload path mismatch | Ensure the `node_tools` directory exists in the same folder as this notebook |
| Script fails with `Permission denied` | `chmod +x` not applied before execution | Ensure the `chmod` command is in the same `add_post_boot_execute()` call |
| Package installation fails in script | Wrong package manager for the OS image | Match `yum`/`dnf` for Rocky/CentOS, `apt` for Ubuntu |
| Slice takes very long to become active | Many post boot tasks or slow package installs | Post boot tasks run after provisioning; complex scripts add time |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_site()` | Get a random FABRIC site name | [get_random_site](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_site) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `node.add_post_boot_upload_directory(local, remote)` | Queue a directory upload for post boot | [add_post_boot_upload_directory](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_post_boot_upload_directory) |
| `node.add_post_boot_execute(command)` | Queue a command for post boot execution | [add_post_boot_execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_post_boot_execute) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you know how to use post boot tasks, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Post Boot Task Templates** | [post_boot_task_templates](../post_boot_task_templates/post_boot_task_templates.ipynb) | Use Jinja2 templates to pass dynamic node-specific info (like interface names) to post boot scripts |
| **Upload and Execute** | [upload_and_execute](../upload_and_execute/upload_and_execute.ipynb) | Manually upload, execute, and download files (the non-automated approach) |
| **Parallel Configuration** | [parallel_config](../parallel_config/parallel_config.ipynb) | Use threads to configure nodes in parallel after they are active |
| **Docker Containers** | [docker_containers](../docker_containers/docker_containers.ipynb) | Deploy Docker containers using post boot tasks |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |